# Day 37: Chi-Square & ANOVA

## Chi-Square Test of Independence
This test answers a very common business question phrased in categorical terms: "does exposure to an ad relate to whether someone buys?" - both "saw an ad" and "bought" are categories (yes/no), not numbers, so a t-test or correlation wouldn't apply here. The chi-square test of independence works from a contingency table, which cross-tabulates counts for every combination of the two categorical variables, and compares those observed counts to what we'd expect to see if the two variables were completely unrelated (independent). Below, the table shows how many people bought vs. didn't, split by whether they saw the ad or not; `chi2_contingency` takes that table and returns the chi-square statistic, a p-value, degrees of freedom, and the "expected" counts it compared against. A p-value close to 0 suggests the actual counts deviate meaningfully from what independence would predict - i.e., ad exposure and purchasing look related.

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# Contingency table: rows = whether they saw the ad, columns = whether they bought
table = pd.DataFrame({
    'Bought': [50, 30],
    'Not_Bought': [20, 40]
}, index=['Saw_Ad', 'No_Ad'])
print(table)
# chi2_contingency compares observed counts to what independence would predict;
# returns the test statistic, p-value, degrees of freedom, and expected counts
chi2, p, dof, expected = chi2_contingency(table)
print('p-value:', p)

## Interpreting the Result
Every hypothesis test starts from a "null hypothesis" - here, that the two categorical variables are independent (no real relationship) - and the p-value tells you how likely it would be to see data this extreme (or more extreme) purely by chance if that null hypothesis were actually true. By convention, a p-value below 0.05 (the significance level, or alpha) is treated as strong enough evidence to reject the null hypothesis and conclude there's a statistically significant relationship; a p-value of 0.05 or above means we don't have enough evidence to say the variables are related, though that doesn't prove they're unrelated. The code below simply applies that standard 0.05 cutoff to the p-value calculated in the previous cell and prints a plain-language conclusion.

In [ ]:
# Standard significance threshold of 0.05: p < 0.05 means "statistically significant"
if p < 0.05:
    print('Significant association between ad exposure and purchase')
else:
    print('No significant association found')

## One-Way ANOVA
A t-test can only compare the means of two groups, but real experiments often involve three or more (e.g. testing three different website layouts, or comparing sales across three regions) - running repeated t-tests on every pair would inflate the chance of a false positive. ANOVA (Analysis of Variance) solves this by testing all groups at once: its null hypothesis is that all group means are equal, and it works by comparing the variance *between* group means to the variance *within* each group - if groups differ a lot more from each other than their own members differ internally, that's evidence the means aren't all the same. `f_oneway` takes each group's raw values as separate arguments and returns the F-statistic (a ratio of between-group to within-group variance - larger values mean bigger relative differences between groups) and a p-value. Below, three groups of numeric scores are compared; a small p-value would mean at least one group's mean is genuinely different from the others, though ANOVA alone doesn't say which one.

In [ ]:
from scipy.stats import f_oneway

# Three independent groups of numeric measurements to compare
group_a = [23, 25, 28, 22, 27]
group_b = [30, 32, 29, 35, 31]
group_c = [20, 21, 19, 23, 22]
# f_oneway tests whether all three group means are equal (the null hypothesis)
f_stat, p_value = f_oneway(group_a, group_b, group_c)
print('F-statistic:', f_stat)
print('p-value:', p_value)

## When to Use Which Test
Picking the right hypothesis test comes down to two questions: what type is the outcome variable (numeric or categorical), and how many groups are you comparing? A t-test is for comparing the mean of a numeric variable between exactly two groups (e.g. average spend: control vs. treatment). ANOVA extends that to three or more groups with a numeric outcome (e.g. average spend across three regions). A chi-square test is for when both variables being compared are categorical, rather than comparing a numeric average (e.g. whether purchase yes/no relates to ad exposure yes/no, as tested earlier in this notebook). Keeping this decision tree in mind - two groups vs. many groups, numeric vs. categorical - avoids picking the wrong test and drawing the wrong conclusion from valid data.

In [ ]:
# Quick reference: which test applies to which situation
print('t-test -> compare 2 group means')
print('ANOVA -> compare 3+ group means')
print('Chi-square -> test association between categories')